In [1]:
import socket
print(socket.gethostname())

awr-2-08


In [2]:
import warnings

import xarray as xr
import hdf5plugin
import matplotlib.pyplot as plt

In [3]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=xr.SerializationWarning)
    dsp = xr.open_mfdataset('/cw3e/mead/projects/cwp167/moerfani_data/regional/2019/01P/wwrf_reanalysis_modellev_d01_2019-01-01*.nc', engine='h5netcdf', data_vars='all')
    dss = xr.open_mfdataset('/cw3e/mead/projects/cwp167/moerfani_data/regional/2019/01S/wwrf_reanalysis_singlelev_d01_2019-01-01*.nc', engine='h5netcdf', data_vars='all')

We will extract the pressure variables and single level variables from all 24 nc files representing one day as follows. These variables will then be resampled to 6-hour intervals and stored in a new netcdf file. For the precipitation variable, this resampling should be different (e.g., accumulation over the period).

**Pressure variables:**
- Geopotential (Z_e)
- Temperature (T_e)
- Specific humidity (q_e)
- Zonal wind (u_gr_e)
- Meridional wind (v_gr_e)

**Single levels:**
- Mean sea level pressure (slp)
- Surface pressure (p_sfc)
- 2-m dewpoint temperature (Td_2m)
- 2-m temperature (T_2m)
- Skin/sea surface temperature (T_sfc)
- Integrated water vapor (IWV)
- Integrated vapor transport (meridional) (IVTV)
- Integrated vapor transport (zonal) (IVTU)
- 10-m zonal wind (u_10m_gr)
- 10-m meridional wind (v_10m_gr)
- 6-hourly accumulated precipitation (precip_bkt)

**Forcing:**
- Geopotential at the surface (Z_sfc)

In [4]:
dsp

<xarray.Dataset> Size: 167GB
Dimensions:            (time: 24, eta: 99, south_north: 970, west_east: 1389)
Coordinates:
  * time               (time) datetime64[ns] 192B 2019-01-01 ... 2019-01-01T2...
  * eta                (eta) float32 396B 0.998 0.9932 ... 0.001282 0.0004168
  * south_north        (south_north) float32 4kB -2.827e+03 ... 2.988e+03
  * west_east          (west_east) float32 6kB -5.238e+03 ... 3.091e+03
    lat                (south_north, west_east) float32 5MB dask.array<chunksize=(970, 1389), meta=np.ndarray>
    lon                (south_north, west_east) float32 5MB dask.array<chunksize=(970, 1389), meta=np.ndarray>
Data variables: (12/21)
    DateTime           (time) int32 96B dask.array<chunksize=(1,), meta=np.ndarray>
    T_e                (time, eta, south_north, west_east) float32 13GB dask.array<chunksize=(1, 99, 970, 1389), meta=np.ndarray>
    quantization_info  (time) |S1 24B b'' b'' b'' b'' b'' ... b'' b'' b'' b''
    Z_e                (time, eta, south_north, west_east) float32 13GB dask.array<chunksize=(1, 99, 970, 1389), meta=np.ndarray>
    Z_sfc              (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 970, 1389), meta=np.ndarray>
    day                (time) int32 96B dask.array<chunksize=(1,), meta=np.ndarray>
    ...                 ...
    r_rain             (time, eta, south_north, west_east) float32 13GB dask.array<chunksize=(1, 99, 970, 1389), meta=np.ndarray>
    r_snow             (time, eta, south_north, west_east) float32 13GB dask.array<chunksize=(1, 99, 970, 1389), meta=np.ndarray>
    u_gr_e             (time, eta, south_north, west_east) float32 13GB dask.array<chunksize=(1, 99, 970, 1389), meta=np.ndarray>
    v_gr_e             (time, eta, south_north, west_east) float32 13GB dask.array<chunksize=(1, 99, 970, 1389), meta=np.ndarray>
    w_e                (time, eta, south_north, west_east) float32 13GB dask.array<chunksize=(1, 99, 970, 1389), meta=np.ndarray>
    year               (time) int32 96B dask.array<chunksize=(1,), meta=np.ndarray>
Attributes:
    CDI:            Climate Data Interface version 2.4.4 (https://mpimet.mpg....
    Conventions:    CF 1.6, Standard Name Table v19
    source:         /cw3e/mead/projects/cnt117/reanalysis_temp/input/2018/d01...
    institution:    CW3E - Scripps Institution of Oceanography
    title:          /cw3e/mead/projects/cnt117/reanalysis_temp/output/2018/d0...
    notes:          Created with NCL script:  wrfout_to_cf.ncl v2.0
    created_by:     Daniel Steinhoff - dsteinhoff@ucsd.edu
    creation_date:  Fri Dec  5 01:18:13 PM PST 2025
    history:        Fri Dec  5 13:24:15 2025: /home/cw3eprod/miniforge3/envs/...
    CDO:            Climate Data Operators version 2.4.4 (https://mpimet.mpg....
    NCO:            netCDF Operators version 5.2.9 (Homepage = http://nco.sf....

In [ ]:
dss

<xarray.Dataset> Size: 9GB
Dimensions:            (time: 24, south_north: 970, west_east: 1389, soil: 4)
Coordinates:
  * time               (time) datetime64[ns] 192B 2019-01-01 ... 2019-01-01T2...
  * south_north        (south_north) float32 4kB -2.827e+03 ... 2.988e+03
  * west_east          (west_east) float32 6kB -5.238e+03 ... 3.091e+03
    lat                (south_north, west_east) float32 5MB dask.array<chunksize=(970, 1389), meta=np.ndarray>
    lon                (south_north, west_east) float32 5MB dask.array<chunksize=(970, 1389), meta=np.ndarray>
  * soil               (soil) float32 16B 0.04999 0.25 0.7002 1.5
Data variables: (12/74)
    AET                (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    ALBEDO_ALL         (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 970, 1389), meta=np.ndarray>
    ALBEDO_BCK         (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    ALBEDO_SFC         (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    CAPE               (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    CIN                (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    ...                 ...
    v_150m_gr          (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    v_30m_gr           (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    v_80m_gr           (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    wd_10m             (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    ws_10m             (time, south_north, west_east) float32 129MB dask.array<chunksize=(1, 855, 1226), meta=np.ndarray>
    year               (time) int32 96B dask.array<chunksize=(1,), meta=np.ndarray>
Attributes:
    CDI:            Climate Data Interface version 2.4.4 (https://mpimet.mpg....
    Conventions:    CF 1.6, Standard Name Table v19
    source:         /data/projects/40YearReanalysis/cwp103/2023.03.01_copy_fr...
    institution:    CW3E - Scripps Institution of Oceanography
    title:          /data/projects/40YearReanalysis/v2/output/singlelev/d01/2...
    notes:          Created with NCL script:  wrfout_to_cf.ncl v2.0
    created_by:     Daniel Steinhoff - dsteinhoff@ucsd.edu
    creation_date:  Tue Jan 20 20:10:24 PST 2026
    NCO:            netCDF Operators version 5.3.2 (Homepage = http://nco.sf....
    CDO:            Climate Data Operators version 2.4.4 (https://mpimet.mpg....

In [ ]:
# Extract pressure variables from dsp
pressure_vars = ['Z_e', 'T_e', 'q_e', 'u_gr_e', 'v_gr_e']
dsp_selected = dsp[pressure_vars]

# Extract single level and forcing variables from dss
single_vars = ['slp', 'p_sfc', 'Td_2m', 'T_2m', 'T_sfc', 'IWV', 'IVTV', 'IVTU', 'u_10m_gr', 'v_10m_gr', 'precip_bkt', 'Z_sfc']
dss_selected = dss[single_vars]

# Merge the datasets
ds_combined = xr.merge([dsp_selected, dss_selected])

# Resample to 6-hour intervals
# For precipitation, use sum (accumulation over the period)
precip_resampled = ds_combined['precip_bkt'].resample(time='6h').sum()

# For other variables, use mean
others = ds_combined.drop_vars('precip_bkt')
others_resampled = others.resample(time='6h').mean()

# Merge back the resampled data
ds_resampled = xr.merge([others_resampled, precip_resampled])

# Save to a new NetCDF file
save_path = '/cw3e/mead/projects/cwp167/moerfani_data/regional/2019/01M/wwrf_reanalysis_modellev_d01_2019-01-01.nc'
ds_resampled.to_netcdf(save_path)

In [6]:
dsm = xr.open_mfdataset(save_path, engine='h5netcdf', data_vars='all')
dsm

<xarray.Dataset> Size: 11GB
Dimensions:      (time: 4, eta: 99, south_north: 970, west_east: 1389)
Coordinates:
  * time         (time) datetime64[ns] 32B 2019-01-01 ... 2019-01-01T18:00:00
  * eta          (eta) float32 396B 0.998 0.9932 0.9873 ... 0.001282 0.0004168
  * south_north  (south_north) float32 4kB -2.827e+03 -2.821e+03 ... 2.988e+03
  * west_east    (west_east) float32 6kB -5.238e+03 -5.232e+03 ... 3.091e+03
    lat          (south_north, west_east) float32 5MB dask.array<chunksize=(970, 1389), meta=np.ndarray>
    lon          (south_north, west_east) float32 5MB dask.array<chunksize=(970, 1389), meta=np.ndarray>
Data variables: (12/17)
    Z_e          (time, eta, south_north, west_east) float32 2GB dask.array<chunksize=(4, 99, 970, 1389), meta=np.ndarray>
    T_e          (time, eta, south_north, west_east) float32 2GB dask.array<chunksize=(4, 99, 970, 1389), meta=np.ndarray>
    q_e          (time, eta, south_north, west_east) float32 2GB dask.array<chunksize=(4, 99, 970, 1389), meta=np.ndarray>
    u_gr_e       (time, eta, south_north, west_east) float32 2GB dask.array<chunksize=(4, 99, 970, 1389), meta=np.ndarray>
    v_gr_e       (time, eta, south_north, west_east) float32 2GB dask.array<chunksize=(4, 99, 970, 1389), meta=np.ndarray>
    slp          (time, south_north, west_east) float32 22MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
    ...           ...
    IVTV         (time, south_north, west_east) float32 22MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
    IVTU         (time, south_north, west_east) float32 22MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
    u_10m_gr     (time, south_north, west_east) float32 22MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
    v_10m_gr     (time, south_north, west_east) float32 22MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
    Z_sfc        (time, south_north, west_east) float32 22MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
    precip_bkt   (time, south_north, west_east) float32 22MB dask.array<chunksize=(4, 970, 1389), meta=np.ndarray>
Attributes:
    CDI:            Climate Data Interface version 2.4.4 (https://mpimet.mpg....
    Conventions:    CF 1.6, Standard Name Table v19
    source:         /cw3e/mead/projects/cnt117/reanalysis_temp/input/2018/d01...
    institution:    CW3E - Scripps Institution of Oceanography
    title:          /cw3e/mead/projects/cnt117/reanalysis_temp/output/2018/d0...
    notes:          Created with NCL script:  wrfout_to_cf.ncl v2.0
    created_by:     Daniel Steinhoff - dsteinhoff@ucsd.edu
    creation_date:  Fri Dec  5 01:18:13 PM PST 2025
    history:        Fri Dec  5 13:24:15 2025: /home/cw3eprod/miniforge3/envs/...
    CDO:            Climate Data Operators version 2.4.4 (https://mpimet.mpg....
    NCO:            netCDF Operators version 5.2.9 (Homepage = http://nco.sf....